In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import exposure # Para CLAHE en color

In [ ]:
# --- Funciones Auxiliares para Mostrar Imágenes ---
def mostrar_imagen(titulo, imagen, cmap=None):
    plt.figure(figsize=(8, 6))
    if cmap:
        plt.imshow(imagen, cmap=cmap)
    else:
        # OpenCV carga en BGR, Matplotlib espera RGB
        plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def mostrar_histograma(imagen, titulo="Histograma"):
    if len(imagen.shape) == 3: # Imagen a color
        color = ('b', 'g', 'r')
        for i, col in enumerate(color):
            hist = cv2.calcHist([imagen], [i], None, [256], [0, 256])
            plt.plot(hist, color=col)
        plt.title(f"{titulo} (RGB)")
    else: # Imagen en escala de grises
        hist = cv2.calcHist([imagen], [0], None, [256], [0, 256])
        plt.plot(hist, color='gray')
        plt.title(f"{titulo} (Escala de Grises)")
    plt.xlim([0, 256])
    plt.show()

In [ ]:
# --- Etapa 1: Adquisición y Comprensión Inicial ---
print("--- Etapa 1: Adquisición y Comprensión Inicial ---")
ruta_imagen = 'mural_desgastado.jpg' # Cambia esto a la ruta de tu imagen
try:
    img_original = cv2.imread(ruta_imagen)
    if img_original is None:
        raise FileNotFoundError(f"No se pudo cargar la imagen en la ruta: {ruta_imagen}")
except FileNotFoundError as e:
    print(e)
    exit()
except Exception as e:
    print(f"Ocurrió un error al cargar la imagen: {e}")
    exit()

mostrar_imagen("Mural Original", img_original)
print(f"Dimensiones de la imagen: {img_original.shape}") # (alto, ancho, canales)
print(f"Tipo de datos: {img_original.dtype}")
mostrar_histograma(img_original, "Histograma Original")

In [ ]:
# --- Etapa 2: Preprocesamiento: Corrección y Preparación ---
print("\n--- Etapa 2: Preprocesamiento ---")

# 2.1 Corrección Gamma (Ajuste de Luminosidad)
# Una gamma > 1 aclara, < 1 oscurece.
gamma = 1.5
tabla_gamma = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                        for i in np.arange(0, 256)]).astype("uint8")
img_gamma_corregida = cv2.LUT(img_original, tabla_gamma)
mostrar_imagen("Corrección Gamma", img_gamma_corregida)
mostrar_histograma(img_gamma_corregida, "Histograma Post-Gamma")

# 2.2 Reducción de Ruido
# Filtro de Mediana (bueno para ruido "sal y pimienta")
img_mediana = cv2.medianBlur(img_gamma_corregida, 5) # El kernel debe ser impar
mostrar_imagen("Filtro de Mediana", img_mediana)

# Filtro Gaussiano (suavizado general)
img_gaussiana = cv2.GaussianBlur(img_mediana, (5, 5), 0)
mostrar_imagen("Filtro Gaussiano", img_gaussiana)

img_preprocesada = img_gaussiana.copy()

In [ ]:
# --- Etapa 3: Realce y Recuperación de Detalles ---
print("\n--- Etapa 3: Realce y Recuperación de Detalles ---")

# 3.1 Ecualización de Histograma Adaptativa (CLAHE) para mejorar contraste local
# Convertir a L*a*b* para aplicar CLAHE solo al canal de luminancia (L)
img_lab = cv2.cvtColor(img_preprocesada, cv2.COLOR_BGR2LAB)
l_channel, a_channel, b_channel = cv2.split(img_lab)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
l_channel_clahe = clahe.apply(l_channel)

img_lab_clahe = cv2.merge((l_channel_clahe, a_channel, b_channel))
img_clahe = cv2.cvtColor(img_lab_clahe, cv2.COLOR_LAB2BGR)
mostrar_imagen("Contraste Mejorado (CLAHE)", img_clahe)
mostrar_histograma(img_clahe, "Histograma Post-CLAHE")

# 3.2 Realce de Bordes (Máscara de Desenfoque - Unsharp Masking)
# Forma simple: original + (original - suavizado) = 2*original - suavizado
# O usando un kernel de realce
blurred_for_sharpen = cv2.GaussianBlur(img_clahe, (0,0), 3) # Kernel más pequeño para no perder el efecto CLAHE
img_realzada = cv2.addWeighted(img_clahe, 1.5, blurred_for_sharpen, -0.5, 0)
# kernel_sharpen = np.array([[-1, -1, -1],
#                            [-1,  9, -1],
#                            [-1, -1, -1]])
# img_realzada = cv2.filter2D(img_clahe, -1, kernel_sharpen)
mostrar_imagen("Realce de Detalles (Sharpening)", img_realzada)

img_mejorada = img_realzada.copy()

In [ ]:
# --- Etapa 4: Segmentación y Aislamiento de Áreas de Daño (Ejemplo Básico) ---
# Esta etapa es muy dependiente de la imagen.
# Aquí simularemos una máscara de daño para el inpainting.
# En un caso real, podrías usar umbralización, detección de bordes, color, etc.
print("\n--- Etapa 4: Segmentación de Áreas de Daño (Simulación) ---")

# Convertir a escala de grises para detección de bordes o umbralización simple
img_gris_para_mascara = cv2.cvtColor(img_mejorada, cv2.COLOR_BGR2GRAY)

# Ejemplo: Umbralización para encontrar áreas muy oscuras (simulando daño)
# Esto es solo un ejemplo, la detección de "daño" real es mucho más compleja.
_, mascara_dano_simple = cv2.threshold(img_gris_para_mascara, 50, 255, cv2.THRESH_BINARY_INV)
mostrar_imagen("Máscara de Daño (Simulada por Umbral)", mascara_dano_simple, cmap='gray')

# Detección de bordes Canny (para identificar contornos, podría ayudar a definir áreas)
bordes_canny = cv2.Canny(img_gris_para_mascara, threshold1=50, threshold2=150)
mostrar_imagen("Detección de Bordes (Canny)", bordes_canny, cmap='gray')

# Para el inpainting, necesitamos una máscara en blanco y negro donde las áreas
# a restaurar sean blancas (255).
# Aquí crearemos una máscara de ejemplo manualmente para demostrar el inpainting.
# En un proyecto real, esta máscara se obtendría de una segmentación más sofisticada
# o incluso dibujada manualmente.

# Creemos una máscara de ejemplo (ej. una grieta simulada o un área)
# Esto es muy rudimentario, solo para probar el inpainting.
mascara_inpaint = np.zeros(img_mejorada.shape[:2], dtype=np.uint8)
# Dibujar algunas "áreas dañadas" en la máscara
cv2.rectangle(mascara_inpaint, (img_mejorada.shape[1]//4, img_mejorada.shape[0]//4),
              (img_mejorada.shape[1]//2, img_mejorada.shape[0]//2), 255, -1)
cv2.line(mascara_inpaint, (10, 10), (img_mejorada.shape[1]-50, img_mejorada.shape[0]-50), 255, 15)
mostrar_imagen("Máscara de Inpainting (Ejemplo Manual)", mascara_inpaint, cmap='gray')

In [ ]:
# --- Etapa 5: Técnicas de Restauración (Inpainting Básico) ---
# OpenCV ofrece dos algoritmos de inpainting.
# cv2.INPAINT_TELEA: Basado en el método de Telea "An Image Inpainting Technique Based on the Fast Marching Method".
# cv2.INPAINT_NS: Basado en el método de Navier-Stokes.
print("\n--- Etapa 5: Restauración (Inpainting) ---")

# Usamos la máscara manual 'mascara_inpaint' que creamos
# El tercer parámetro es el radio de la vecindad alrededor de un píxel que se usa para el inpainting.
img_restaurada_telea = cv2.inpaint(img_mejorada, mascara_inpaint, 5, cv2.INPAINT_TELEA)
mostrar_imagen("Imagen Restaurada (Inpainting Telea)", img_restaurada_telea)

img_restaurada_ns = cv2.inpaint(img_mejorada, mascara_inpaint, 5, cv2.INPAINT_NS)
mostrar_imagen("Imagen Restaurada (Inpainting NS)", img_restaurada_ns)

# Elegimos una para continuar (Telea suele ser más rápido, NS a veces mejor para texturas)
img_final_restaurada = img_restaurada_telea.copy()

In [ ]:
# --- Etapa 6: Ajustes Finales y Guardado ---
print("\n--- Etapa 6: Ajustes Finales y Guardado ---")
# Aquí podrías aplicar ajustes finales de color, brillo, etc. de forma más sutil
# o realizar retoques manuales si esto fuera parte de un flujo interactivo.

# Ejemplo de un pequeño ajuste de brillo final si es necesario
# img_final_ajustada = cv2.convertScaleAbs(img_final_restaurada, alpha=1.0, beta=10) # alpha: contraste, beta: brillo
# mostrar_imagen("Imagen Final con Ajuste de Brillo", img_final_ajustada)
# img_para_guardar = img_final_ajustada

img_para_guardar = img_final_restaurada

# Guardar la imagen resultante
ruta_salida_tiff = 'mural_restaurado.tiff' # Formato sin pérdida
ruta_salida_jpeg = 'mural_restaurado.jpg'  # Formato con pérdida (alta calidad)

try:
    cv2.imwrite(ruta_salida_tiff, img_para_guardar)
    cv2.imwrite(ruta_salida_jpeg, img_para_guardar, [cv2.IMWRITE_JPEG_QUALITY, 95])
    print(f"Imagen restaurada guardada como '{ruta_salida_tiff}' y '{ruta_salida_jpeg}'")
except Exception as e:
    print(f"Error al guardar la imagen: {e}")

print("\n--- Proceso Completado ---")